# Week 87, Day 3: Architecture and design review

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w087d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('waymark/domain.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\n\n@dataclass(frozen=True)\nclass Junction:\n    name: str\n    height_m: int\n\ndef naismith_minutes(km, climb_m):\n    return round(km / 5 * 60 + climb_m / 600 * 60)\n')
with open('waymark/routing.py', 'w', encoding='utf-8') as f:
    f.write('import heapq\nfrom waymark.domain import Junction, naismith_minutes\nfrom waymark.storage import load_paths\n\ndef shortest(paths, start, end):\n    ...\n')
with open('waymark/storage.py', 'w', encoding='utf-8') as f:
    f.write('import sqlite3\nfrom waymark.domain import Junction\n\ndef load_paths(conn):\n    ...\n')
with open('waymark/services.py', 'w', encoding='utf-8') as f:
    f.write('from waymark import routing\nfrom waymark.storage import load_paths, save_route\n\ndef plan(conn, start, end):\n    ...\n')
with open('waymark/api.py', 'w', encoding='utf-8') as f:
    f.write('import json\nfrom waymark.services import plan\nfrom waymark.storage import save_route\n\ndef handle(method, path, body):\n    ...\n')
with open('waymark/cli.py', 'w', encoding='utf-8') as f:
    f.write('import argparse\nfrom waymark.services import plan\n\ndef main(argv):\n    ...\n')

**Predict before you run:** what will this print?



In [ ]:
import ast

source = '''
import json
import waymark.storage
from waymark.domain import Junction
from . import helpers
'''
tree = ast.parse(source)
for node in ast.walk(tree):
    if isinstance(node, ast.Import):
        print("import", [a.name for a in node.names])
    elif isinstance(node, ast.ImportFrom):
        print("from", node.module, "level", node.level, [a.name for a in node.names])

In [ ]:
import ast

def project_imports(source, project="waymark"):
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names if a.name.split(".")[0] == project}
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module.split(".")[0] == project:
            found.add(node.module)
    return found

for name in ["domain", "routing", "api"]:
    print(name, sorted(project_imports(open(f"waymark/{name}.py").read())))

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("PRAGMA foreign_keys = ON")
conn.executescript('''
CREATE TABLE junction (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, height_m INTEGER NOT NULL);
CREATE TABLE path (id INTEGER PRIMARY KEY,
                   start INTEGER NOT NULL REFERENCES junction(id),
                   finish INTEGER NOT NULL REFERENCES junction(id),
                   km REAL NOT NULL CHECK (km > 0));
INSERT INTO junction (name, height_m) VALUES ('Mill', 40), ('Ridge', 310);
''')
for row in [(1, 2, 3.5), (1, 9, 2.0), (1, 2, -1.0)]:
    try:
        conn.execute("INSERT INTO path (start, finish, km) VALUES (?, ?, ?)", row)
        print(row, "stored")
    except sqlite3.IntegrityError as e:
        print(row, "rejected:", e)

> **Key idea.** Divide an application into layers: a pure domain, storage, services (one per user action, owning transactions) and interfaces (API, CLI). Dependencies point inwards; interfaces go through services. Check the rule automatically by reading imports with `ast`, mapping modules to layers and comparing with an allowed table; find import cycles with a depth-first search that tracks the current path. Turn the data model into a schema with primary keys, foreign keys (with `PRAGMA foreign_keys = ON` in SQLite) and constraints. Map every route to a story. Record significant decisions as short, never-edited ADRs (title, status, context, decision, consequences). Review the design against a checklist before building.

### Exercise: Reading imports

Write `imports_of(source, project)`: the set of **project** modules a source file imports: for `import a.b`, the name `a.b`; for `from a.b import c`, the name `a.b`, or `a.b.c` when `a.b.c` is in the optional set `modules` (because then `c` is a module, as in `from waymark import routing`). Only names whose first part is `project` count; ignore relative imports (`from . import x`). Signature: `imports_of(source, project, modules=())`.

In [ ]:
import ast

def imports_of(source, project, modules=()):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_imports():\n    \"Import and ImportFrom\"\n    mods = {\"waymark.domain\", \"waymark.routing\", \"waymark.storage\", \"waymark.services\"}\n    src = \"import json\\nimport waymark.storage\\nfrom waymark.domain import Junction\\nfrom waymark import routing\\nfrom . import helpers\\nimport waymarker\\n\"\n    assert imports_of(src, \"waymark\", mods) == {\"waymark.storage\", \"waymark.domain\", \"waymark.routing\"}\n    assert imports_of(src, \"waymark\") == {\"waymark.storage\", \"waymark.domain\", \"waymark\"}\n    assert imports_of(open(\"waymark/services.py\").read(), \"waymark\", mods) == {\"waymark.routing\", \"waymark.storage\"}\n    nested = \"def f():\\n    from waymark.storage import save_route\\n    return save_route\\n\"\n    assert imports_of(nested, \"waymark\", mods) == {\"waymark.storage\"}, \"imports inside functions count too\"")

### Exercise: Checking the layers

Write `violations(files, layers, allowed, project)`: `files` maps module names (like `"waymark.api"`) to source; `layers` maps module names to layer names; `allowed` maps each layer to the set of layers it may import. Using the given `imports_of` (with `modules` set to the keys of `files`), return a sorted list of `(importer, imported, importer_layer, imported_layer)` for every import that breaks the rule. Imported modules not in `layers` are ignored.

In [ ]:
import ast

def imports_of(source, project, modules=()):
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names if a.name.split(".")[0] == project}
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module and node.module.split(".")[0] == project:
            subs = {f"{node.module}.{a.name}" for a in node.names} & set(modules)
            found |= subs if subs else {node.module}
    return found

def violations(files, layers, allowed, project):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\n\nLAYERS = {\"waymark.domain\": \"domain\", \"waymark.routing\": \"domain\", \"waymark.storage\": \"storage\",\n          \"waymark.services\": \"service\", \"waymark.api\": \"interface\", \"waymark.cli\": \"interface\"}\nALLOWED = {\"domain\": {\"domain\"}, \"storage\": {\"storage\", \"domain\"},\n           \"service\": {\"service\", \"storage\", \"domain\"}, \"interface\": {\"interface\", \"service\", \"domain\"}}\n\ndef _files():\n    return {f\"waymark.{name[:-3]}\": open(f\"waymark/{name}\").read() for name in sorted(os.listdir(\"waymark\"))}\n\ndef test_layers():\n    \"the two shortcuts in the skeleton\"\n    assert violations(_files(), LAYERS, ALLOWED, \"waymark\") == [\n        (\"waymark.api\", \"waymark.storage\", \"interface\", \"storage\"),\n        (\"waymark.routing\", \"waymark.storage\", \"domain\", \"storage\")]\n    fixed = _files()\n    fixed[\"waymark.routing\"] = fixed[\"waymark.routing\"].replace(\"from waymark.storage import load_paths\\n\", \"\")\n    fixed[\"waymark.api\"] = fixed[\"waymark.api\"].replace(\"from waymark.storage import save_route\\n\", \"\")\n    assert violations(fixed, LAYERS, ALLOWED, \"waymark\") == []")

### Exercise: Finding import cycles

Write `find_cycle(graph)`: `graph` maps each module to the set of modules it imports. Return one cycle as a list that starts and ends with the same module (like `["a", "b", "c", "a"]`), or `None`. Search deterministically: start from modules in sorted order and visit dependencies in sorted order, returning the first cycle found; a module reached again while it's **on the current path** closes a cycle (reaching a module that's already finished is fine).

In [ ]:
def find_cycle(graph):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cycles():\n    \"on the path, not merely seen\"\n    assert find_cycle({\"a\": {\"b\"}, \"b\": {\"c\"}, \"c\": {\"a\"}}) == [\"a\", \"b\", \"c\", \"a\"]\n    assert find_cycle({\"api\": {\"services\", \"domain\"}, \"services\": {\"domain\", \"storage\"}, \"storage\": {\"domain\"}, \"domain\": set()}) is None\n    assert find_cycle({\"a\": {\"b\", \"c\"}, \"b\": {\"d\"}, \"c\": {\"d\"}, \"d\": set()}) is None, \"a diamond isn't a cycle\"\n    assert find_cycle({\"x\": {\"y\"}, \"y\": {\"z\"}, \"z\": {\"y\"}}) == [\"y\", \"z\", \"y\"]\n    assert find_cycle({\"self\": {\"self\"}}) == [\"self\", \"self\"]\n    assert find_cycle({\"a\": {\"missing\"}}) is None")

### Exercise: A schema from a data model

Write `create_table(name, fields)`: `fields` is a list of `(column, type, rules)`, where `rules` is a list of strings: `"pk"` (`PRIMARY KEY`), `"required"` (`NOT NULL`), `"unique"` (`UNIQUE`), `"ref TABLE"` (`REFERENCES TABLE(id)`) and `"check EXPR"` (`CHECK (column EXPR)`). Return the statement as `CREATE TABLE name (` then each column on its own line, indented four spaces, as `column TYPE` followed by its rules in the order given, separated by commas, then `);`. Raise `ValueError("unknown rule R")` for anything else.

In [ ]:
import sqlite3

def create_table(name, fields):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import sqlite3\n\ndef test_schema():\n    \"keys and constraints\"\n    junction = create_table(\"junction\", [(\"id\", \"INTEGER\", [\"pk\"]), (\"name\", \"TEXT\", [\"required\", \"unique\"]),\n                                         (\"height_m\", \"INTEGER\", [\"required\", \"check >= -500\"])])\n    assert junction == (\"CREATE TABLE junction (\\n    id INTEGER PRIMARY KEY,\\n    name TEXT NOT NULL UNIQUE,\\n\"\n                        \"    height_m INTEGER NOT NULL CHECK (height_m >= -500)\\n);\")\n    path = create_table(\"path\", [(\"id\", \"INTEGER\", [\"pk\"]), (\"start\", \"INTEGER\", [\"required\", \"ref junction\"]),\n                                 (\"km\", \"REAL\", [\"required\", \"check > 0\"])])\n    conn = sqlite3.connect(\":memory:\")\n    conn.execute(\"PRAGMA foreign_keys = ON\")\n    conn.execute(junction)\n    conn.execute(path)\n    conn.execute(\"INSERT INTO junction (name, height_m) VALUES ('Mill', 40)\")\n    conn.execute(\"INSERT INTO path (start, km) VALUES (1, 2.5)\")\n    for bad in (\"INSERT INTO path (start, km) VALUES (7, 2.5)\", \"INSERT INTO path (start, km) VALUES (1, 0)\",\n                \"INSERT INTO junction (name, height_m) VALUES ('Mill', 10)\"):\n        try:\n            conn.execute(bad)\n        except sqlite3.IntegrityError:\n            pass\n        else:\n            raise AssertionError(bad)\n    try:\n        create_table(\"x\", [(\"a\", \"TEXT\", [\"indexed\"])])\n    except ValueError as e:\n        assert str(e) == \"unknown rule indexed\"")

### Exercise: Routes and stories

`routes` is a list of `(method, path, stories)`, with `stories` a list of ids. Write `uncovered(story_ids, routes)`: the story ids served by no route, in order. And `route_problems(routes)`: a sorted list of problems: `"verb in path: METHOD PATH"` for any path segment (ignoring the query string) that is one of `VERBS` (given) or starts with one followed by `-`; `"duplicate: METHOD PATH"` for a method and path that appear more than once (reported once); and `"no story: METHOD PATH"` for a route serving no story.

In [ ]:
VERBS = {"get", "create", "add", "delete", "remove", "update", "find", "save", "list"}

def uncovered(story_ids, routes):
    ___

def route_problems(routes):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_api():\n    \"every story served, every route explained\"\n    routes = [(\"GET\", \"/routes?from=A&to=C\", [\"S1\", \"S2\"]), (\"POST\", \"/saved-routes\", [\"S3\"]),\n              (\"GET\", \"/saved-routes\", [\"S3\"]), (\"POST\", \"/walks\", [\"S5\"])]\n    assert uncovered([\"S1\", \"S2\", \"S3\", \"S4\", \"S5\"], routes) == [\"S4\"]\n    assert route_problems(routes) == []\n    messy = routes + [(\"POST\", \"/create-walk\", [\"S5\"]), (\"GET\", \"/walks/find\", []), (\"POST\", \"/walks\", [\"S5\"])]\n    assert route_problems(messy) == [\"duplicate: POST /walks\", \"no story: GET /walks/find\",\n                                     \"verb in path: GET /walks/find\", \"verb in path: POST /create-walk\"]")

### Exercise: Decision records

An ADR looks like the example in the checks: a first line `# ADR N: Title`, a line `Status: STATUS` (`proposed`, `accepted`, `rejected`, or `superseded by N`), then `## Context`, `## Decision` and `## Consequences` sections. Write `parse_adr(text)`: a dict with `number` (int), `title`, `status`, and `context`, `decision` and `consequences` (each section's text, stripped); raise `ValueError("missing SECTION")` for the first missing section, checked in that order. Then `index(adrs)`: lines `ADR N: Title (status)` sorted by number, for a list of parsed ADRs, with superseded ones shown as `(superseded by N)`, and `current(adrs)`: the numbers of accepted ADRs that nothing supersedes.

In [ ]:
import re

def parse_adr(text):
    ___

def index(adrs):
    ___

def current(adrs):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("ONE = \"\"\"# ADR 1: Use SQLite for storage\nStatus: accepted\n\n## Context\nOne user, one computer, a few thousand rows.\n\n## Decision\nStore everything in a single SQLite file.\n\n## Consequences\nNo server to run. Only one writer at a time.\n\"\"\"\nTWO = \"\"\"# ADR 2: Use a JSON file for saved routes\nStatus: superseded by 3\n\n## Context\nSaved routes are few.\n\n## Decision\nKeep them in routes.json.\n\n## Consequences\nTwo storage mechanisms.\n\"\"\"\nTHREE = \"\"\"# ADR 3: Keep saved routes in SQLite too\nStatus: accepted\n\n## Context\nTwo storage mechanisms caused bugs.\n\n## Decision\nMove saved routes into the database.\n\n## Consequences\nOne place for all data.\n\"\"\"\n\ndef test_adrs():\n    \"parse, index, current\"\n    one = parse_adr(ONE)\n    assert one == {\"number\": 1, \"title\": \"Use SQLite for storage\", \"status\": \"accepted\",\n                   \"context\": \"One user, one computer, a few thousand rows.\",\n                   \"decision\": \"Store everything in a single SQLite file.\",\n                   \"consequences\": \"No server to run. Only one writer at a time.\"}\n    adrs = [parse_adr(t) for t in (THREE, ONE, TWO)]\n    assert index(adrs) == [\"ADR 1: Use SQLite for storage (accepted)\", \"ADR 2: Use a JSON file for saved routes (superseded by 3)\",\n                           \"ADR 3: Keep saved routes in SQLite too (accepted)\"]\n    assert current(adrs) == [1, 3]\n    try:\n        parse_adr(ONE.replace(\"## Decision\", \"## Choice\"))\n    except ValueError as e:\n        assert str(e) == \"missing Decision\"")

### Bug hunt: Imports that weren't seen

The checker only looks at `import` statements, so it misses `from waymark.storage import load_paths` and prints `['waymark.domain']`. Handle `from ... import` as well, so it prints `['waymark.domain', 'waymark.storage']`.

In [ ]:
import ast

source = "import waymark.domain\nfrom waymark.storage import load_paths\n"
found = set()
for node in ast.walk(ast.parse(source)):
    if isinstance(node, ast.Import):
        found |= {a.name for a in node.names}
print(sorted(found))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ImportFrom too\"\n    expect_output(\"['waymark.domain', 'waymark.storage']\")")

### Bug hunt: A diamond mistaken for a cycle

`api` and `cli` both use `domain`, which is fine, but the search treats any module it has **ever** seen as a cycle: it prints `cycle at domain`. Track the modules on the **current path** separately from those already finished, so it prints `no cycle`.

In [ ]:
graph = {"app": ["api", "cli"], "api": ["domain"], "cli": ["domain"], "domain": []}
seen = set()

def visit(node):
    if node in seen:
        return node
    seen.add(node)
    for nxt in graph[node]:
        found = visit(nxt)
        if found:
            return found
    return None

found = visit("app")
print(f"cycle at {found}" if found else "no cycle")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"on the path, not merely seen\"\n    expect_output(\"no cycle\")")

### Bug hunt: A foreign key that wasn't checked

SQLite doesn't enforce foreign keys unless asked, so a path to a junction that doesn't exist is stored: it prints `stored`. Turn enforcement on, so it prints `rejected`.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE junction (id INTEGER PRIMARY KEY, name TEXT)")
conn.execute("CREATE TABLE path (id INTEGER PRIMARY KEY, start INTEGER REFERENCES junction(id))")
try:
    conn.execute("INSERT INTO path (start) VALUES (42)")
    print("stored")
except sqlite3.IntegrityError:
    print("rejected")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"PRAGMA foreign_keys\"\n    expect_output(\"rejected\")")

### Puzzle: An interface that goes through a service

Arrange Waymark's API handler for saving a route so that it only translates: parse the request, call the service, turn the result or error into a response. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the layer that holds Naismith's rule: "domain", "storage", "service" or "interface"
q = ___   # the layer that owns database transactions for "save a walk"
r = ___   # True or False: an HTTP handler should run SQL itself
s = ___   # the SQLite statement (without the semicolon) that switches foreign keys on
t = ___   # what you write when an accepted ADR's decision changes: "edit it" or "a new ADR"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"domain\" and q == \"service\" and r is False, \"p, q and r\"\n    assert s.upper().replace(\" \", \"\") == \"PRAGMAFOREIGN_KEYS=ON\" and t == \"a new ADR\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: an architecture check

Write `architecture_report(files, layers, allowed, project, entry_points)` for a project's modules (`files` maps module names to source). Using the given helpers, return a dict with: `graph` (each module, sorted, to the sorted list of project modules it imports, keeping only modules in `files`); `violations` (as in the exercise); `cycle` (from `find_cycle`, or `None`); `unused` (sorted modules that no other module imports and that aren't entry points); `fan_in` and `fan_out` (dicts from each module to how many modules import it and how many it imports); `unlayered` (sorted modules in `files` with no layer); and `ok` (`True` only if there are no violations, no cycle, nothing unused and nothing unlayered).

In [ ]:
import ast

def imports_of(source, project, modules=()):
    found = set()
    for node in ast.walk(ast.parse(source)):
        if isinstance(node, ast.Import):
            found |= {a.name for a in node.names if a.name.split(".")[0] == project}
        elif isinstance(node, ast.ImportFrom) and node.level == 0 and node.module and node.module.split(".")[0] == project:
            subs = {f"{node.module}.{a.name}" for a in node.names} & set(modules)
            found |= subs if subs else {node.module}
    return found

def find_cycle(graph):
    done, path = set(), []
    def visit(node):
        if node in path:
            return path[path.index(node):] + [node]
        if node in done:
            return None
        path.append(node)
        for nxt in sorted(graph.get(node, ())):
            cycle = visit(nxt)
            if cycle:
                return cycle
        path.pop()
        done.add(node)
        return None
    for start in sorted(graph):
        cycle = visit(start)
        if cycle:
            return cycle
    return None

def architecture_report(files, layers, allowed, project, entry_points):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import os\n\nLAYERS = {\"waymark.domain\": \"domain\", \"waymark.routing\": \"domain\", \"waymark.storage\": \"storage\",\n          \"waymark.services\": \"service\", \"waymark.api\": \"interface\", \"waymark.cli\": \"interface\"}\nALLOWED = {\"domain\": {\"domain\"}, \"storage\": {\"storage\", \"domain\"},\n           \"service\": {\"service\", \"storage\", \"domain\"}, \"interface\": {\"interface\", \"service\", \"domain\"}}\nENTRY = {\"waymark.api\", \"waymark.cli\"}\n\ndef _files():\n    return {f\"waymark.{name[:-3]}\": open(f\"waymark/{name}\").read() for name in sorted(os.listdir(\"waymark\"))}\n\ndef test_skeleton():\n    \"the skeleton as it stands\"\n    r = architecture_report(_files(), LAYERS, ALLOWED, \"waymark\", ENTRY)\n    assert r[\"graph\"] == {\"waymark.api\": [\"waymark.services\", \"waymark.storage\"], \"waymark.cli\": [\"waymark.services\"],\n                          \"waymark.domain\": [], \"waymark.routing\": [\"waymark.domain\", \"waymark.storage\"],\n                          \"waymark.services\": [\"waymark.routing\", \"waymark.storage\"], \"waymark.storage\": [\"waymark.domain\"]}\n    assert [v[:2] for v in r[\"violations\"]] == [(\"waymark.api\", \"waymark.storage\"), (\"waymark.routing\", \"waymark.storage\")]\n    assert r[\"cycle\"] is None and r[\"unused\"] == [] and r[\"unlayered\"] == []\n    assert r[\"fan_in\"][\"waymark.storage\"] == 3 and r[\"fan_out\"][\"waymark.services\"] == 2 and r[\"fan_in\"][\"waymark.api\"] == 0\n    assert r[\"ok\"] is False\n\ndef test_problems():\n    \"cycles, orphans and missing layers\"\n    files = _files()\n    files[\"waymark.routing\"] = files[\"waymark.routing\"].replace(\"from waymark.storage import load_paths\\n\", \"\")\n    files[\"waymark.api\"] = files[\"waymark.api\"].replace(\"from waymark.storage import save_route\\n\", \"\")\n    assert architecture_report(files, LAYERS, ALLOWED, \"waymark\", ENTRY)[\"ok\"] is True\n    files[\"waymark.domain\"] += \"\\nfrom waymark.routing import shortest\\n\"\n    files[\"waymark.export\"] = \"import waymark.domain\\n\"\n    r = architecture_report(files, LAYERS, ALLOWED, \"waymark\", ENTRY)\n    assert r[\"cycle\"] == [\"waymark.routing\", \"waymark.domain\", \"waymark.routing\"]\n    assert r[\"unused\"] == [\"waymark.export\"] and r[\"unlayered\"] == [\"waymark.export\"] and r[\"ok\"] is False")

## Warm-up

Python can read its own code as a tree (Week 64's `ast`), so a program's **imports**, and therefore its dependencies, can be checked automatically. Today that turns an architecture diagram from a hope into a test.

## Layers and the dependency rule

Most applications divide naturally into **layers**, each with one job:

- **Domain**: the ideas the program is about and the rules about them: for Waymark, junctions, paths, routes, Naismith's rule and the route search. Plain Python, no files, no database, no network, so it's easy to test and to reason about.
- **Storage**: getting domain objects in and out of the database (the repository pattern, Week 36).
- **Services** (or use cases): one function per thing a user can do ("plan a route", "save a walk"), coordinating domain and storage, and owning transactions.
- **Interfaces**: the ways in: an HTTP API, a command-line tool, later perhaps a web page. They translate requests into service calls and results into responses, and nothing more.

The **dependency rule**: dependencies point **inwards**, towards the domain. Interfaces may use services and the domain; services may use storage and the domain; storage may use the domain; the domain uses none of them. So the domain never changes because the database or the API did, and each outer part can be swapped or tested with a fake. Waymark's rule, as data:

```python
LAYERS = {"waymark.domain": "domain", "waymark.routing": "domain", "waymark.storage": "storage",
          "waymark.services": "service", "waymark.api": "interface", "waymark.cli": "interface"}
ALLOWED = {"domain": {"domain"}, "storage": {"storage", "domain"},
           "service": {"service", "storage", "domain"}, "interface": {"interface", "service", "domain"}}
```

Notice the interfaces aren't allowed to reach past the services into storage: if the API saved routes directly, the rules in the service (validation, transactions) would be skipped.

## Checking the rule

To check the rule, read each module's imports (`ast.Import` and `ast.ImportFrom` nodes), keep those that are modules of your project, and compare each importer's layer with the imported module's. Look in the files above: the skeleton already has two violations, of the kind that creep in when you're in a hurry. `routing.py`, in the domain, imports storage; `api.py` calls storage directly.

**Cycles** are the other thing to look for: `a` imports `b` imports `a`. They make modules impossible to understand (or import) separately, and in Python they can fail at import time. A depth-first search that tracks which modules are **on the current path** finds them (Week 49's cycle detection in directed graphs).

`from waymark import routing` names the package `waymark`; a full checker also looks at the imported names to see that `routing` is a module (the challenge does).

## The data model and the API

The **data model** lists the things the program stores and how they relate. Waymark's: a `junction` (a name and a height), a `path` between two junctions (length, climb each way), a saved `route` (a name and an ordered list of junctions), and a `walk` (a route, a date, how long it actually took). Turn it into a **schema** with:

- a **primary key** on every table;
- **foreign keys** for relationships (`path.start` references `junction.id`), switched on in SQLite with `PRAGMA foreign_keys = ON`;
- **constraints** that encode rules: `NOT NULL`, `UNIQUE` (no two routes with the same name), `CHECK (km > 0)`.

Constraints in the database are a second line of defence: even a bug in the code can't store a path of negative length.

The **API** gets a table too: method, path, what it does, which story it serves, and its success and error codes. Every must-story should have a route (or a CLI command); every route should serve a story. Plural nouns for collections, no verbs in paths (Week 77).

| Method | Path | Story | Success | Errors |
|---|---|---|---|---|
| GET | /routes?from=A&to=C | S1, S2 | 200 | 400, 404 |
| POST | /saved-routes | S3 | 201 | 400, 409 |
| GET | /saved-routes | S3 | 200 | |
| GET | /round-walks?from=A&minutes=120 | S4 | 200 | 400 |
| POST | /walks | S5 | 201 | 400, 404 |
| GET | /walks/summary?year=2026 | S5 | 200 | 400 |

## Decisions and the review

Some decisions are worth recording: anything that was hard to decide, would be expensive to change, or that someone will later ask "why?" about. An **architecture decision record** (ADR) is a short file per decision (`docs/adr/0003-use-sqlite.md`) with a **title**, a **status** (proposed, accepted, superseded by another), the **context** (the forces at play), the **decision**, and its **consequences** (good and bad). They're never edited after acceptance; a changed mind is a new ADR that supersedes the old one, so the history stays readable.

Then hold a **design review**: give someone the spec, the layer diagram, the schema, the API table and the ADRs, and ask them to break it. A useful checklist:

- Does every must-story have a home (a service, a route or command, a table)?
- Do dependencies point inwards? Are there cycles?
- What happens when things fail: no route, a duplicate name, a database error halfway through?
- Can the data get into an impossible state? Which constraints stop it?
- Where's the riskiest part, and when will it be built? (Waymark's: route search on a big map, built in week one.)
- How will each part be tested, and what needs a fake?
- What's the simplest version of this that would still work?

## Practice

## Quiz

## Daily challenge

## Your capstone today

On your own computer, in your capstone's `docs/` folder:

1. **Layers**: list your modules (even if they're empty files yet) and which layer each belongs to, as a `LAYERS` dict like Waymark's, with your `ALLOWED` table. Save today's `architecture_report` as `tools/check_architecture.py`; on Day 4 it becomes a CI step.
2. **Data model**: every table, its keys and constraints, as `create_table` calls or plain SQL in `docs/schema.sql`.
3. **Interface table**: every route or command, with the story it serves, its success and error results. Run `uncovered` against your stories.
4. **Two or three ADRs** in `docs/adr/`: at least your storage choice and your interface choice.
5. **A design review**: send the spec and these documents to someone (a friend from the course is ideal) with the checklist from today's lesson, and ask for their three biggest worries. Write each down with what you'll do about it.

## Before you go

Find the riskiest part of your design and write down when you'll build it. If the answer is "week four", move it to this week's first slice. Tomorrow is **repository, CI and skeleton**: the project's home, set up so that every change is checked from the first commit.